Environment Setup

In [1]:
# ============================================================
# CELL 1 — Environment & GPU Verification
# ============================================================

import os
import json
import time
import random
import platform
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print("=" * 60)
print("ENVIRONMENT CHECK")
print("=" * 60)

print("TensorFlow version :", tf.__version__)
print("Python version     :", platform.python_version())
print("Platform           :", platform.platform())
print("Timestamp          :", datetime.now().isoformat())

gpus = tf.config.list_physical_devices("GPU")

print("\nGPU count:", len(gpus))

for i, gpu in enumerate(gpus):
    print(f"GPU {i}: {gpu}")

print("=" * 60)

if len(gpus) == 0:
    print("WARNING: No GPU detected.")
else:
    print(f"SUCCESS: {len(gpus)} GPU(s) detected.")

ENVIRONMENT CHECK
TensorFlow version : 2.20.0
Python version     : 3.12.13
Platform           : Linux-6.12.90+-x86_64-with-glibc2.35
Timestamp          : 2026-08-19T04:53:21.462995

GPU count: 2
GPU 0: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
GPU 1: PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
SUCCESS: 2 GPU(s) detected.


In [2]:
# ============================================================
# CELL 2 — Detailed GPU Hardware Record
# ============================================================

print("=" * 60)
print("NVIDIA GPU HARDWARE")
print("=" * 60)

!nvidia-smi --query-gpu=index,name,memory.total,memory.free,memory.used,utilization.gpu --format=csv

print("=" * 60)

NVIDIA GPU HARDWARE
index, name, memory.total [MiB], memory.free [MiB], memory.used [MiB], utilization.gpu [%]
0, Tesla T4, 15360 MiB, 14909 MiB, 3 MiB, 0 %
1, Tesla T4, 15360 MiB, 14909 MiB, 3 MiB, 0 %


In [3]:
# ============================================================
# CELL 3 — Experiment Configuration
# ============================================================

CONFIG = {
    # Experiment identity
    "experiment_name": "AE_280K_full",
    "dataset_name": "evilsocket/alucard-sprites",

    # Image configuration
    "image_height": 128,
    "image_width": 128,
    "channels": 4,

    # Dataset target
    # Actual usable size will be determined after deduplication.
    "dataset_size_target": 280000,

    # Split ratios
    "train_ratio": 0.90,
    "validation_ratio": 0.05,
    "test_ratio": 0.05,

    # Training
    # We will decide the final batch size after checking
    # the original 10K configuration and benchmarking.
    "batch_size_per_gpu": None,
    "epochs": 50,

    # Optimization
    "learning_rate": 1e-3,
    "optimizer": "Adam",
    "loss": "mse",

    # Reproducibility
    "seed": 42,

    # Checkpointing
    "checkpoint_every_epoch": True,
}

print("=" * 60)
print("EXPERIMENT CONFIGURATION")
print("=" * 60)

for key, value in CONFIG.items():
    print(f"{key}: {value}")

print("=" * 60)

EXPERIMENT CONFIGURATION
experiment_name: AE_280K_full
dataset_name: evilsocket/alucard-sprites
image_height: 128
image_width: 128
channels: 4
dataset_size_target: 280000
train_ratio: 0.9
validation_ratio: 0.05
test_ratio: 0.05
batch_size_per_gpu: None
epochs: 50
learning_rate: 0.001
optimizer: Adam
loss: mse
seed: 42
checkpoint_every_epoch: True


In [4]:
# ============================================================
# CELL 4 — Reproducibility & Runtime Metadata
# ============================================================

SEED = CONFIG["seed"]

# Python / NumPy / TensorFlow seeds
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Store runtime information in CONFIG
CONFIG["tensorflow_version"] = tf.__version__
CONFIG["python_version"] = platform.python_version()
CONFIG["platform"] = platform.platform()
CONFIG["timestamp"] = datetime.now().isoformat()

# Record detected GPUs
CONFIG["gpu_count"] = len(tf.config.list_physical_devices("GPU"))
CONFIG["gpus"] = [
    str(gpu) for gpu in tf.config.list_physical_devices("GPU")
]

print("=" * 60)
print("REPRODUCIBILITY & RUNTIME METADATA")
print("=" * 60)

print("Seed               :", SEED)
print("TensorFlow         :", CONFIG["tensorflow_version"])
print("Python             :", CONFIG["python_version"])
print("GPU count          :", CONFIG["gpu_count"])

for i, gpu in enumerate(CONFIG["gpus"]):
    print(f"GPU {i}              :", gpu)

print("Timestamp           :", CONFIG["timestamp"])

print("=" * 60)

REPRODUCIBILITY & RUNTIME METADATA
Seed               : 42
TensorFlow         : 2.20.0
Python             : 3.12.13
GPU count          : 2
GPU 0              : PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
GPU 1              : PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')
Timestamp           : 2026-08-19T04:53:22.056684


In [5]:
# ============================================================
# CELL 5 — Output Directory Structure
# ============================================================

BASE_DIR = "/kaggle/working/AE_280K"

DIRS = {
    "base": BASE_DIR,
    "checkpoints": os.path.join(BASE_DIR, "checkpoints"),
    "models": os.path.join(BASE_DIR, "models"),
    "logs": os.path.join(BASE_DIR, "logs"),
    "evaluation": os.path.join(BASE_DIR, "evaluation"),
    "config": os.path.join(BASE_DIR, "config"),
    "samples": os.path.join(BASE_DIR, "samples"),
}

# Create directories
for directory in DIRS.values():
    os.makedirs(directory, exist_ok=True)

print("=" * 60)
print("OUTPUT DIRECTORIES")
print("=" * 60)

for name, path in DIRS.items():
    print(f"{name:15s}: {path}")

print("=" * 60)

# Store the output location in the experiment configuration
CONFIG["output_directory"] = BASE_DIR

OUTPUT DIRECTORIES
base           : /kaggle/working/AE_280K
checkpoints    : /kaggle/working/AE_280K/checkpoints
models         : /kaggle/working/AE_280K/models
logs           : /kaggle/working/AE_280K/logs
evaluation     : /kaggle/working/AE_280K/evaluation
config         : /kaggle/working/AE_280K/config
samples        : /kaggle/working/AE_280K/samples


DATASET

In [6]:
# ============================================================
# CELL 6 — Load Alucard Sprites Dataset
# ============================================================

# Install the Hugging Face datasets library only if needed
try:
    from datasets import load_dataset
    print("Hugging Face 'datasets' library is already installed.")
except ImportError:
    print("Installing Hugging Face 'datasets' library...")
    !pip install -q datasets
    from datasets import load_dataset

print("\nLoading dataset:")
print(CONFIG["dataset_name"])

dataset = load_dataset(CONFIG["dataset_name"])

print("\n" + "=" * 60)
print("DATASET LOADED")
print("=" * 60)

print(dataset)

print("\nDataset splits:")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data):,} samples")

print("=" * 60)

Hugging Face 'datasets' library is already installed.

Loading dataset:
evilsocket/alucard-sprites

DATASET LOADED
DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 312550
    })
})

Dataset splits:
  train: 312,550 samples


In [7]:
# ============================================================
# CELL 7 — Verify Dataset Structure & Image Format
# ============================================================

print("=" * 60)
print("DATASET VERIFICATION")
print("=" * 60)

train_data = dataset["train"]

# Basic dataset information
print("Number of samples :", len(train_data))
print("Features          :", train_data.features)

# Inspect first sample
sample = train_data[0]
image = sample["image"]

print("\nFirst sample:")
print("Image type        :", type(image))
print("Image size        :", image.size)
print("Image mode        :", image.mode)
print("Text              :", repr(sample["text"]))

# Convert to NumPy only for this ONE sample
image_array = np.asarray(image)

print("\nImage array:")
print("Shape             :", image_array.shape)
print("Dtype             :", image_array.dtype)
print("Min pixel value   :", image_array.min())
print("Max pixel value   :", image_array.max())

# Verify expected format
assert image.size == (
    CONFIG["image_width"],
    CONFIG["image_height"]
), f"Unexpected image size: {image.size}"

assert image.mode == "RGBA", f"Unexpected image mode: {image.mode}"

print("\n✓ Image size verified: 128 × 128")
print("✓ Image mode verified: RGBA")
print("✓ Dataset structure verified")

print("=" * 60)

DATASET VERIFICATION
Number of samples : 312550
Features          : {'image': Image(mode=None, decode=True), 'text': Value('string')}

First sample:
Image type        : <class 'PIL.PngImagePlugin.PngImageFile'>
Image size        : (128, 128)
Image mode        : RGBA
Text              : 'pixel art, gray, small, wizard, mage, spellcaster, back view'

Image array:
Shape             : (128, 128, 4)
Dtype             : uint8
Min pixel value   : 0
Max pixel value   : 255

✓ Image size verified: 128 × 128
✓ Image mode verified: RGBA
✓ Dataset structure verified


Resumable exact-image deduplication

In [8]:
# ============================================================
# CELL 8 — Resumable Exact-Image Deduplication
# ============================================================

import hashlib

DEDUP_STATE_PATH = os.path.join(
    DIRS["config"],
    "dedup_state.json"
)

UNIQUE_INDICES_PATH = os.path.join(
    DIRS["config"],
    "unique_indices.json"
)


def image_hash(image):
    """
    Create an exact SHA-256 hash from the RGBA pixel data.
    Images with identical pixels will have identical hashes.
    """
    pixels = np.asarray(
        image.convert("RGBA"),
        dtype=np.uint8
    )

    return hashlib.sha256(pixels.tobytes()).hexdigest()


# ------------------------------------------------------------
# Check whether deduplication was already completed
# ------------------------------------------------------------

if os.path.exists(UNIQUE_INDICES_PATH):

    print("=" * 60)
    print("DEDUPLICATION ALREADY COMPLETED")
    print("=" * 60)

    with open(UNIQUE_INDICES_PATH, "r") as f:
        dedup_data = json.load(f)

    unique_indices = dedup_data["unique_indices"]

    print(f"Unique images loaded: {len(unique_indices):,}")
    print(
        f"Duplicates removed : "
        f"{len(train_data) - len(unique_indices):,}"
    )

else:

    # --------------------------------------------------------
    # Resume an interrupted deduplication run if possible
    # --------------------------------------------------------

    if os.path.exists(DEDUP_STATE_PATH):

        print("=" * 60)
        print("RESUMING DEDUPLICATION")
        print("=" * 60)

        with open(DEDUP_STATE_PATH, "r") as f:
            state = json.load(f)

        start_index = state["next_index"]
        unique_indices = state["unique_indices"]
        seen_hashes = set(state["seen_hashes"])

        print(f"Resuming from image : {start_index:,}")
        print(f"Unique images so far : {len(unique_indices):,}")

    else:

        print("=" * 60)
        print("STARTING DEDUPLICATION")
        print("=" * 60)

        start_index = 0
        unique_indices = []
        seen_hashes = set()

        print(f"Total images: {len(train_data):,}")

    # --------------------------------------------------------
    # Process images
    # --------------------------------------------------------

    CHECKPOINT_INTERVAL = 10_000

    for i in range(start_index, len(train_data)):

        image = train_data[i]["image"]

        h = image_hash(image)

        if h not in seen_hashes:
            seen_hashes.add(h)
            unique_indices.append(i)

        # Save progress periodically
        if (i + 1) % CHECKPOINT_INTERVAL == 0:

            state = {
                "next_index": i + 1,
                "unique_indices": unique_indices,
                "seen_hashes": list(seen_hashes)
            }

            with open(DEDUP_STATE_PATH, "w") as f:
                json.dump(state, f)

            print(
                f"Processed: {i + 1:,} / {len(train_data):,} "
                f"| Unique: {len(unique_indices):,}"
            )

    # --------------------------------------------------------
    # Deduplication completed
    # --------------------------------------------------------

    dedup_data = {
        "unique_indices": unique_indices,
        "total_source_images": len(train_data),
        "unique_images": len(unique_indices),
        "duplicates_removed": (
            len(train_data) - len(unique_indices)
        ),
        "completed_at": datetime.now().isoformat()
    }

    with open(UNIQUE_INDICES_PATH, "w") as f:
        json.dump(dedup_data, f, indent=4)

    # Remove temporary resumable state
    if os.path.exists(DEDUP_STATE_PATH):
        os.remove(DEDUP_STATE_PATH)

    print("\n" + "=" * 60)
    print("DEDUPLICATION COMPLETE")
    print("=" * 60)

    print(f"Source images      : {len(train_data):,}")
    print(f"Unique images      : {len(unique_indices):,}")
    print(
        f"Exact duplicates   : "
        f"{len(train_data) - len(unique_indices):,}"
    )

    print("=" * 60)

DEDUPLICATION ALREADY COMPLETED
Unique images loaded: 282,511
Duplicates removed : 30,039


In [9]:
# ============================================================
# CELL 9 — Deterministic Shuffle of Unique Dataset
# ============================================================

print("=" * 60)
print("PREPARING UNIQUE DATASET ORDER")
print("=" * 60)

# Load the completed deduplication result
with open(UNIQUE_INDICES_PATH, "r") as f:
    dedup_data = json.load(f)

unique_indices = dedup_data["unique_indices"]

print(f"Unique images available: {len(unique_indices):,}")

# ------------------------------------------------------------
# Create a deterministic shuffled ordering
# ------------------------------------------------------------

SHUFFLED_INDICES_PATH = os.path.join(
    DIRS["config"],
    "shuffled_unique_indices.json"
)

if os.path.exists(SHUFFLED_INDICES_PATH):

    print("\nExisting shuffled ordering found.")
    print("Loading saved ordering...")

    with open(SHUFFLED_INDICES_PATH, "r") as f:
        shuffle_data = json.load(f)

    shuffled_unique_indices = shuffle_data["shuffled_unique_indices"]

else:

    print("\nCreating deterministic shuffled ordering...")

    rng = np.random.default_rng(SEED)

    shuffled_unique_indices = np.array(
        unique_indices,
        dtype=np.int64
    )

    rng.shuffle(shuffled_unique_indices)

    shuffled_unique_indices = shuffled_unique_indices.tolist()

    shuffle_data = {
        "seed": SEED,
        "total_unique_images": len(shuffled_unique_indices),
        "shuffled_unique_indices": shuffled_unique_indices,
        "created_at": datetime.now().isoformat()
    }

    with open(SHUFFLED_INDICES_PATH, "w") as f:
        json.dump(shuffle_data, f, indent=4)

    print("Shuffled ordering saved.")

# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert len(shuffled_unique_indices) == len(unique_indices)

assert len(set(shuffled_unique_indices)) == len(
    shuffled_unique_indices
)

print("\n" + "=" * 60)
print("SHUFFLE VERIFIED")
print("=" * 60)

print(
    f"Unique images : {len(shuffled_unique_indices):,}"
)
print(
    f"Seed          : {SEED}"
)
print(
    "Ordering      : deterministic and saved"
)

print("=" * 60)

PREPARING UNIQUE DATASET ORDER
Unique images available: 282,511

Existing shuffled ordering found.
Loading saved ordering...

SHUFFLE VERIFIED
Unique images : 282,511
Seed          : 42
Ordering      : deterministic and saved


In [10]:
# ============================================================
# CELL 10 — Deterministic Train / Validation / Test Split
# ============================================================

print("=" * 60)
print("CREATING TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)


# ------------------------------------------------------------
# Load the deterministic shuffled unique indices
# ------------------------------------------------------------

with open(SHUFFLED_INDICES_PATH, "r") as f:
    shuffle_data = json.load(f)

shuffled_unique_indices = shuffle_data["shuffled_unique_indices"]

TOTAL_CLEAN = len(shuffled_unique_indices)

print(f"Total unique images : {TOTAL_CLEAN:,}")

# ------------------------------------------------------------
# Calculate exact split sizes from the actual cleaned dataset
# ------------------------------------------------------------

TRAIN_SIZE = int(round(
    TOTAL_CLEAN * CONFIG["train_ratio"]
))

VAL_SIZE = int(round(
    TOTAL_CLEAN * CONFIG["validation_ratio"]
))

TEST_SIZE = TOTAL_CLEAN - TRAIN_SIZE - VAL_SIZE

# Safety checks
assert TRAIN_SIZE > 0
assert VAL_SIZE > 0
assert TEST_SIZE > 0

assert (
    TRAIN_SIZE + VAL_SIZE + TEST_SIZE
    == TOTAL_CLEAN
)

print("\nSplit sizes:")
print(f"Training   : {TRAIN_SIZE:,}")
print(f"Validation : {VAL_SIZE:,}")
print(f"Test       : {TEST_SIZE:,}")

print("\nPercentages:")
print(
    f"Training   : {TRAIN_SIZE / TOTAL_CLEAN * 100:.4f}%"
)
print(
    f"Validation : {VAL_SIZE / TOTAL_CLEAN * 100:.4f}%"
)
print(
    f"Test       : {TEST_SIZE / TOTAL_CLEAN * 100:.4f}%"
)

# ------------------------------------------------------------
# Create actual manifests
# These contain ORIGINAL Hugging Face dataset indices.
# ------------------------------------------------------------

train_indices = shuffled_unique_indices[:TRAIN_SIZE]

val_indices = shuffled_unique_indices[
    TRAIN_SIZE:
    TRAIN_SIZE + VAL_SIZE
]

test_indices = shuffled_unique_indices[
    TRAIN_SIZE + VAL_SIZE:
]

# ------------------------------------------------------------
# Verify no overlap
# ------------------------------------------------------------

train_set = set(train_indices)
val_set = set(val_indices)
test_set = set(test_indices)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

assert (
    len(train_indices)
    + len(val_indices)
    + len(test_indices)
    == TOTAL_CLEAN
)

print("\n✓ No train/validation/test overlap")
print("✓ Every unique image assigned exactly once")

# ------------------------------------------------------------
# Save individual manifests
# ------------------------------------------------------------

TRAIN_MANIFEST_PATH = os.path.join(
    DIRS["config"],
    "train_indices.json"
)

VAL_MANIFEST_PATH = os.path.join(
    DIRS["config"],
    "val_indices.json"
)

TEST_MANIFEST_PATH = os.path.join(
    DIRS["config"],
    "test_indices.json"
)

with open(TRAIN_MANIFEST_PATH, "w") as f:
    json.dump(train_indices, f)

with open(VAL_MANIFEST_PATH, "w") as f:
    json.dump(val_indices, f)

with open(TEST_MANIFEST_PATH, "w") as f:
    json.dump(test_indices, f)

# ------------------------------------------------------------
# Save complete dataset manifest
# ------------------------------------------------------------

DATASET_MANIFEST_PATH = os.path.join(
    DIRS["config"],
    "dataset_manifest.json"
)

dataset_manifest = {
    "dataset_name": CONFIG["dataset_name"],
    "seed": SEED,

    "source_images": len(train_data),

    "unique_images": TOTAL_CLEAN,
    "duplicates_removed": (
        len(train_data) - TOTAL_CLEAN
    ),

    "train_size": TRAIN_SIZE,
    "validation_size": VAL_SIZE,
    "test_size": TEST_SIZE,

    "train_ratio": TRAIN_SIZE / TOTAL_CLEAN,
    "validation_ratio": VAL_SIZE / TOTAL_CLEAN,
    "test_ratio": TEST_SIZE / TOTAL_CLEAN,

    "train_indices_file": "train_indices.json",
    "validation_indices_file": "val_indices.json",
    "test_indices_file": "test_indices.json",

    "created_at": datetime.now().isoformat()
}

with open(DATASET_MANIFEST_PATH, "w") as f:
    json.dump(dataset_manifest, f, indent=4)

# ------------------------------------------------------------
# Update CONFIG with actual dataset sizes
# ------------------------------------------------------------

CONFIG["actual_clean_dataset_size"] = TOTAL_CLEAN
CONFIG["train_size"] = TRAIN_SIZE
CONFIG["validation_size"] = VAL_SIZE
CONFIG["test_size"] = TEST_SIZE

print("\n" + "=" * 60)
print("SPLIT COMPLETE")
print("=" * 60)

print(f"Clean dataset : {TOTAL_CLEAN:,}")
print(f"Train         : {TRAIN_SIZE:,}")
print(f"Validation    : {VAL_SIZE:,}")
print(f"Test          : {TEST_SIZE:,}")

print("\nSaved manifests:")
print(f"  {TRAIN_MANIFEST_PATH}")
print(f"  {VAL_MANIFEST_PATH}")
print(f"  {TEST_MANIFEST_PATH}")
print(f"  {DATASET_MANIFEST_PATH}")

print("=" * 60)

CREATING TRAIN / VALIDATION / TEST SPLIT
Total unique images : 282,511

Split sizes:
Training   : 254,260
Validation : 14,126
Test       : 14,125

Percentages:
Training   : 90.0000%
Validation : 5.0002%
Test       : 4.9998%

✓ No train/validation/test overlap
✓ Every unique image assigned exactly once

SPLIT COMPLETE
Clean dataset : 282,511
Train         : 254,260
Validation    : 14,126
Test          : 14,125

Saved manifests:
  /kaggle/working/AE_280K/config/train_indices.json
  /kaggle/working/AE_280K/config/val_indices.json
  /kaggle/working/AE_280K/config/test_indices.json
  /kaggle/working/AE_280K/config/dataset_manifest.json


In [11]:
# ============================================================
# CELL 11 — Lazy tf.data Input Pipeline
# ============================================================

print("=" * 60)
print("BUILDING LAZY DATA PIPELINE")
print("=" * 60)

# ------------------------------------------------------------
# Load saved split manifests
# ------------------------------------------------------------

with open(TRAIN_MANIFEST_PATH, "r") as f:
    train_indices = json.load(f)

with open(VAL_MANIFEST_PATH, "r") as f:
    val_indices = json.load(f)

with open(TEST_MANIFEST_PATH, "r") as f:
    test_indices = json.load(f)

print(f"Train indices : {len(train_indices):,}")
print(f"Val indices   : {len(val_indices):,}")
print(f"Test indices  : {len(test_indices):,}")


# ------------------------------------------------------------
# Image preprocessing
# ------------------------------------------------------------

def preprocess_image(image):
    """
    Convert a dataset image into the exact format expected
    by the autoencoder.

    Output:
        float32 tensor
        shape = (128, 128, 4)
        values = [0, 1]
    """

    image = image.convert("RGBA")

    image = np.asarray(
        image,
        dtype=np.float32
    )

    image /= 255.0

    return image


# ------------------------------------------------------------
# Python generators
# ------------------------------------------------------------

def image_generator(indices):
    """
    Lazily load images from the Hugging Face dataset.

    Only the current sample is held in memory.
    """

    for index in indices:

        image = train_data[int(index)]["image"]

        image = preprocess_image(image)

        yield image, image


# ------------------------------------------------------------
# Output signature
# ------------------------------------------------------------

OUTPUT_SIGNATURE = (
    tf.TensorSpec(
        shape=(
            CONFIG["image_height"],
            CONFIG["image_width"],
            CONFIG["channels"]
        ),
        dtype=tf.float32
    ),
    tf.TensorSpec(
        shape=(
            CONFIG["image_height"],
            CONFIG["image_width"],
            CONFIG["channels"]
        ),
        dtype=tf.float32
    )
)


# ------------------------------------------------------------
# Create datasets
# ------------------------------------------------------------

train_raw = tf.data.Dataset.from_generator(
    lambda: image_generator(train_indices),
    output_signature=OUTPUT_SIGNATURE
)

val_raw = tf.data.Dataset.from_generator(
    lambda: image_generator(val_indices),
    output_signature=OUTPUT_SIGNATURE
)

test_raw = tf.data.Dataset.from_generator(
    lambda: image_generator(test_indices),
    output_signature=OUTPUT_SIGNATURE
)


# ------------------------------------------------------------
# Initial batch size
#
# We deliberately leave CONFIG batch_size_per_gpu as None
# until the benchmark determines a safe value.
# ------------------------------------------------------------

print("\n✓ Lazy datasets created")
print("✓ No full image arrays created")
print("✓ Images will be loaded batch-by-batch")

print("=" * 60)

BUILDING LAZY DATA PIPELINE
Train indices : 254,260
Val indices   : 14,126
Test indices  : 14,125

✓ Lazy datasets created
✓ No full image arrays created
✓ Images will be loaded batch-by-batch


I0000 00:00:1787115204.631985    1009 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787115204.634187    1009 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [12]:
# ============================================================
# CELL 12 — Input Pipeline Sanity Check
# ============================================================

print("=" * 60)
print("INPUT PIPELINE SANITY CHECK")
print("=" * 60)

# Verify dataset objects exist
print("train_raw exists :", "train_raw" in globals())
print("val_raw exists   :", "val_raw" in globals())
print("test_raw exists  :", "test_raw" in globals())

assert "train_raw" in globals()
assert "val_raw" in globals()
assert "test_raw" in globals()

# ------------------------------------------------------------
# Read ONLY one sample from the training pipeline
# ------------------------------------------------------------

sample_image, sample_target = next(
    iter(train_raw.take(1))
)

print("\nSingle sample:")
print("Image shape      :", sample_image.shape)
print("Target shape     :", sample_target.shape)
print("Image dtype      :", sample_image.dtype)
print("Target dtype     :", sample_target.dtype)

print(
    "Image min        :",
    float(tf.reduce_min(sample_image))
)

print(
    "Image max        :",
    float(tf.reduce_max(sample_image))
)

# ------------------------------------------------------------
# Verify expected format
# ------------------------------------------------------------

expected_shape = (
    CONFIG["image_height"],
    CONFIG["image_width"],
    CONFIG["channels"]
)

assert tuple(sample_image.shape) == expected_shape
assert tuple(sample_target.shape) == expected_shape

assert sample_image.dtype == tf.float32
assert sample_target.dtype == tf.float32

assert float(tf.reduce_min(sample_image)) >= 0.0
assert float(tf.reduce_max(sample_image)) <= 1.0

print("\n✓ Dataset objects exist")
print("✓ One image successfully loaded")
print("✓ Shape is 128 × 128 × 4")
print("✓ dtype is float32")
print("✓ Pixel range is [0, 1]")
print("✓ Lazy pipeline is working")

print("=" * 60)

INPUT PIPELINE SANITY CHECK
train_raw exists : True
val_raw exists   : True
test_raw exists  : True

Single sample:
Image shape      : (128, 128, 4)
Target shape     : (128, 128, 4)
Image dtype      : <dtype: 'float32'>
Target dtype     : <dtype: 'float32'>
Image min        : 0.0
Image max        : 1.0

✓ Dataset objects exist
✓ One image successfully loaded
✓ Shape is 128 × 128 × 4
✓ dtype is float32
✓ Pixel range is [0, 1]
✓ Lazy pipeline is working


Autoencoder

In [13]:
# ============================================================
# CELL 13 — Exact 10K AE + 2-GPU MirroredStrategy
# ============================================================

from tensorflow import keras
from tensorflow.keras import layers

print("=" * 60)
print("MULTI-GPU STRATEGY + AUTOENCODER")
print("=" * 60)

# ------------------------------------------------------------
# Create distributed training strategy
# ------------------------------------------------------------

strategy = tf.distribute.MirroredStrategy()

print("Number of replicas:", strategy.num_replicas_in_sync)

assert strategy.num_replicas_in_sync == 2, (
    f"Expected 2 GPUs, found "
    f"{strategy.num_replicas_in_sync}"
)


# ------------------------------------------------------------
# EXACT 10K AUTOENCODER ARCHITECTURE
# Do not modify this for the first 280K experiment.
# ------------------------------------------------------------

def build_autoencoder():

    encoder = keras.Sequential([
        layers.Input(
            shape=(
                CONFIG["image_height"],
                CONFIG["image_width"],
                CONFIG["channels"]
            )
        ),

        layers.Conv2D(
            32,
            3,
            activation="relu",
            padding="same",
            strides=2
        ),

        layers.Conv2D(
            64,
            3,
            activation="relu",
            padding="same",
            strides=2
        ),

        layers.Conv2D(
            128,
            3,
            activation="relu",
            padding="same",
            strides=2
        ),

        layers.Conv2D(
            256,
            3,
            activation="relu",
            padding="same",
            strides=2
        ),

    ], name="encoder")


    decoder = keras.Sequential([

        layers.Input(
            shape=(8, 8, 256)
        ),

        layers.Conv2DTranspose(
            128,
            4,
            activation="relu",
            padding="same",
            strides=2
        ),

        layers.Conv2DTranspose(
            64,
            4,
            activation="relu",
            padding="same",
            strides=2
        ),

        layers.Conv2DTranspose(
            32,
            4,
            activation="relu",
            padding="same",
            strides=2
        ),

        layers.Conv2DTranspose(
            4,
            4,
            activation="sigmoid",
            padding="same",
            strides=2
        ),

    ], name="decoder")


    autoencoder = keras.Sequential(
        [
            encoder,
            decoder
        ],
        name="autoencoder"
    )

    autoencoder.compile(
        optimizer=keras.optimizers.Adam(
            learning_rate=CONFIG["learning_rate"]
        ),
        loss=CONFIG["loss"]
    )

    return autoencoder, encoder, decoder


# ------------------------------------------------------------
# IMPORTANT:
# Model creation happens INSIDE strategy.scope()
# ------------------------------------------------------------

with strategy.scope():

    autoencoder, encoder, decoder = build_autoencoder()


# ------------------------------------------------------------
# Verify model
# ------------------------------------------------------------

print("\nModel created successfully.")

print("\n" + "=" * 60)
print("AUTOENCODER SUMMARY")
print("=" * 60)

autoencoder.summary()

print("=" * 60)

print(
    f"\nTotal parameters: "
    f"{autoencoder.count_params():,}"
)

print(
    f"Trainable parameters: "
    f"{sum(np.prod(v.shape) for v in autoencoder.trainable_weights):,}"
)

print("\n✓ Exact 10K architecture loaded")
print("✓ Model created inside strategy.scope()")
print("✓ 2-GPU MirroredStrategy confirmed")
print("✓ Optimizer: Adam")
print("✓ Loss: MSE")

print("=" * 60)

MULTI-GPU STRATEGY + AUTOENCODER
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of replicas: 2

Model created successfully.

AUTOENCODER SUMMARY


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder (Sequential)            │ (None, 8, 8, 256)      │       388,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Sequential)            │ (None, 128, 128, 4)    │       690,404 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,079,108 (4.12 MB)

 Trainable params: 1,079,108 (4.12 MB)

 Non-trainable params: 0 (0.00 B)


Total parameters: 1,079,108
Trainable parameters: 1,079,108

✓ Exact 10K architecture loaded
✓ Model created inside strategy.scope()
✓ 2-GPU MirroredStrategy confirmed
✓ Optimizer: Adam
✓ Loss: MSE


In [14]:
# ============================================================
# CELL 14 — Benchmark Dataset Builder
# ============================================================

print("=" * 60)
print("PREPARING BATCHED DATA PIPELINE FOR BENCHMARK")
print("=" * 60)

# Candidate per-GPU batch sizes.
# Global batch size = per-GPU batch × number of replicas.
CANDIDATE_BATCH_SIZES = [8, 16, 32]

print("Number of replicas :", strategy.num_replicas_in_sync)
print(
    "Candidate per-GPU batch sizes :",
    CANDIDATE_BATCH_SIZES
)

print("\nCorresponding global batch sizes:")

for batch_size in CANDIDATE_BATCH_SIZES:
    global_batch = (
        batch_size * strategy.num_replicas_in_sync
    )
    print(
        f"  per-GPU {batch_size:2d}"
        f" → global {global_batch:2d}"
    )


def prepare_dataset(raw_dataset, batch_size, training=False):
    """
    Add batching and prefetching to a lazy dataset.
    """

    ds = raw_dataset

    if training:
        ds = ds.shuffle(
            buffer_size=2048,
            seed=SEED,
            reshuffle_each_iteration=True
        )

    ds = ds.batch(
        batch_size,
        drop_remainder=training
    )

    ds = ds.prefetch(
        tf.data.AUTOTUNE
    )

    return ds


print("\n✓ Benchmark dataset builder ready")
print("✓ Lazy loading preserved")
print("✓ Batching enabled")
print("✓ Prefetching enabled")

print("=" * 60)

PREPARING BATCHED DATA PIPELINE FOR BENCHMARK
Number of replicas : 2
Candidate per-GPU batch sizes : [8, 16, 32]

Corresponding global batch sizes:
  per-GPU  8 → global 16
  per-GPU 16 → global 32
  per-GPU 32 → global 64

✓ Benchmark dataset builder ready
✓ Lazy loading preserved
✓ Batching enabled
✓ Prefetching enabled


In [15]:
# ============================================================
# CELL 15 — Batch Size & Throughput Benchmark
# ============================================================

import gc
import time

print("=" * 60)
print("BATCH SIZE & THROUGHPUT BENCHMARK")
print("=" * 60)

BENCHMARK_BATCHES = 20
BENCHMARK_RESULTS = []

for per_gpu_batch in CANDIDATE_BATCH_SIZES:

    global_batch = (
        per_gpu_batch * strategy.num_replicas_in_sync
    )

    print("\n" + "-" * 60)
    print(
        f"Testing per-GPU batch size : {per_gpu_batch}"
    )
    print(
        f"Global batch size          : {global_batch}"
    )
    print("-" * 60)

    # Clear previous dataset iterator/resources
    gc.collect()

    try:
        benchmark_ds = prepare_dataset(
            train_raw,
            batch_size=global_batch,
            training=False
        )

        # ----------------------------------------------------
        # Warm-up
        # ----------------------------------------------------

        iterator = iter(benchmark_ds)

        warmup_batches = []

        for _ in range(2):
            batch_x, batch_y = next(iterator)
            warmup_batches.append((batch_x, batch_y))

        # ----------------------------------------------------
        # Timed model inference
        # ----------------------------------------------------

        start_time = time.perf_counter()

        processed_images = 0

        for batch_number in range(BENCHMARK_BATCHES):

            batch_x, batch_y = next(iterator)

            # Forward pass through the actual AE
            predictions = autoencoder(batch_x, training=False)

            # Force TensorFlow to materialize the computation
            _ = predictions.numpy()

            processed_images += batch_x.shape[0]

        elapsed = time.perf_counter() - start_time

        images_per_second = (
            processed_images / elapsed
        )

        batches_per_second = (
            BENCHMARK_BATCHES / elapsed
        )

        result = {
            "per_gpu_batch": per_gpu_batch,
            "global_batch": global_batch,
            "processed_images": int(processed_images),
            "elapsed_seconds": elapsed,
            "images_per_second": images_per_second,
            "batches_per_second": batches_per_second,
            "status": "SUCCESS"
        }

        BENCHMARK_RESULTS.append(result)

        print(
            f"Processed images : {processed_images:,}"
        )

        print(
            f"Elapsed time     : {elapsed:.2f} sec"
        )

        print(
            f"Images/sec       : "
            f"{images_per_second:.2f}"
        )

        print(
            f"Batches/sec      : "
            f"{batches_per_second:.2f}"
        )

        print("Status           : SUCCESS")

        # Cleanup
        del benchmark_ds
        del iterator
        del warmup_batches
        del batch_x
        del batch_y
        del predictions

        gc.collect()

    except Exception as e:

        print("Status           : FAILED")
        print("Error            :", repr(e))

        BENCHMARK_RESULTS.append({
            "per_gpu_batch": per_gpu_batch,
            "global_batch": global_batch,
            "processed_images": 0,
            "elapsed_seconds": None,
            "images_per_second": None,
            "batches_per_second": None,
            "status": "FAILED",
            "error": repr(e)
        })

        # Try to clean up before next candidate
        gc.collect()


# ------------------------------------------------------------
# Display benchmark summary
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("BENCHMARK SUMMARY")
print("=" * 60)

benchmark_df = pd.DataFrame(BENCHMARK_RESULTS)

display(benchmark_df)

print("=" * 60)

BATCH SIZE & THROUGHPUT BENCHMARK

------------------------------------------------------------
Testing per-GPU batch size : 8
Global batch size          : 16
------------------------------------------------------------
Processed images : 320
Elapsed time     : 1.45 sec
Images/sec       : 220.70
Batches/sec      : 13.79
Status           : SUCCESS

------------------------------------------------------------
Testing per-GPU batch size : 16
Global batch size          : 32
------------------------------------------------------------
Processed images : 640
Elapsed time     : 1.22 sec
Images/sec       : 525.91
Batches/sec      : 16.43
Status           : SUCCESS

------------------------------------------------------------
Testing per-GPU batch size : 32
Global batch size          : 64
------------------------------------------------------------
Processed images : 1,280
Elapsed time     : 2.01 sec
Images/sec       : 636.01
Batches/sec      : 9.94
Status           : SUCCESS

BENCHMARK SUMMARY

,per_gpu_batch,global_batch,processed_images,elapsed_seconds,images_per_second,batches_per_second,status
0,8,16,320,1.449904,220.704263,13.794016,SUCCESS
1,16,32,640,1.216946,525.906845,16.434589,SUCCESS
2,32,64,1280,2.012538,636.012729,9.937699,SUCCESS


In [16]:
# ============================================================
# CELL 16 — Real Training-Step Benchmark
# ============================================================

import gc
import time

print("=" * 60)
print("REAL TRAINING-STEP BENCHMARK")
print("=" * 60)

BENCHMARK_PER_GPU_BATCH = 32
BENCHMARK_GLOBAL_BATCH = (
    BENCHMARK_PER_GPU_BATCH
    * strategy.num_replicas_in_sync
)

BENCHMARK_STEPS = 10

print(
    f"Per-GPU batch size : {BENCHMARK_PER_GPU_BATCH}"
)
print(
    f"Global batch size  : {BENCHMARK_GLOBAL_BATCH}"
)
print(
    f"Training steps     : {BENCHMARK_STEPS}"
)

# ------------------------------------------------------------
# Create benchmark dataset
# ------------------------------------------------------------

benchmark_train_ds = prepare_dataset(
    train_raw,
    batch_size=BENCHMARK_GLOBAL_BATCH,
    training=True
)

# ------------------------------------------------------------
# Use the actual compiled model.
# IMPORTANT: this performs optimizer updates.
# ------------------------------------------------------------

optimizer_iterations_before = int(
    autoencoder.optimizer.iterations.numpy()
)

print(
    "\nOptimizer iterations before:",
    optimizer_iterations_before
)

iterator = iter(benchmark_train_ds)

# ------------------------------------------------------------
# Warm-up steps
# ------------------------------------------------------------

print("\nRunning warm-up steps...")

for _ in range(2):

    batch_x, batch_y = next(iterator)

    autoencoder.train_on_batch(
        batch_x,
        batch_y
    )

# ------------------------------------------------------------
# Timed training steps
# ------------------------------------------------------------

print("Running timed training steps...")

start_time = time.perf_counter()

losses = []

for step in range(BENCHMARK_STEPS):

    batch_x, batch_y = next(iterator)

    loss = autoencoder.train_on_batch(
        batch_x,
        batch_y
    )

    # Keras may return a scalar or list
    if isinstance(loss, (list, tuple)):
        loss_value = float(loss[0])
    else:
        loss_value = float(loss)

    losses.append(loss_value)

elapsed = time.perf_counter() - start_time

# ------------------------------------------------------------
# Calculate throughput
# ------------------------------------------------------------

processed_images = (
    BENCHMARK_STEPS
    * BENCHMARK_GLOBAL_BATCH
)

images_per_second = (
    processed_images / elapsed
)

seconds_per_step = (
    elapsed / BENCHMARK_STEPS
)

optimizer_iterations_after = int(
    autoencoder.optimizer.iterations.numpy()
)

print("\n" + "=" * 60)
print("TRAINING BENCHMARK RESULT")
print("=" * 60)

print(
    f"Processed images      : {processed_images:,}"
)

print(
    f"Elapsed time          : {elapsed:.2f} sec"
)

print(
    f"Seconds per step      : {seconds_per_step:.4f}"
)

print(
    f"Training images/sec   : "
    f"{images_per_second:.2f}"
)

print(
    f"Average benchmark loss: "
    f"{np.mean(losses):.6f}"
)

print(
    "Optimizer iterations  : "
    f"{optimizer_iterations_before} → "
    f"{optimizer_iterations_after}"
)

# ------------------------------------------------------------
# Verify optimizer actually updated
# ------------------------------------------------------------

expected_iterations = (
    optimizer_iterations_before
    + BENCHMARK_STEPS
    + 2
)

assert optimizer_iterations_after == expected_iterations

print("\n✓ Forward pass completed")
print("✓ Backpropagation completed")
print("✓ Adam optimizer updates completed")
print("✓ Batch size 32/GPU fits during training")
print("✓ Training-step benchmark successful")

print("=" * 60)

# Cleanup benchmark dataset/iterator
del benchmark_train_ds
del iterator
del batch_x
del batch_y

gc.collect()

REAL TRAINING-STEP BENCHMARK
Per-GPU batch size : 32
Global batch size  : 64
Training steps     : 10

Optimizer iterations before: 0

Running warm-up steps...
INFO:tensorflow:Collective all_reduce tensors: 16 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
Running timed training steps...

TRAINING BENCHMARK RESULT
Processed images      : 640
Elapsed time          : 0.90 sec
Seconds per step      : 0.0901
Training images/sec   : 710.60
Average benchmark loss: 0.198868
Optimizer iterations  : 0 → 12

✓ Forward pass completed
✓ Backpropagation completed
✓ Adam optimizer updates completed
✓ Batch size 32/GPU fits during training
✓ Training-step benchmark successful


0

In [17]:
# ============================================================
# CELL 17 — Lock Validated Training Configuration
# ============================================================

VALIDATED_BATCH_SIZE_PER_GPU = 32

VALIDATED_GLOBAL_BATCH_SIZE = (
    VALIDATED_BATCH_SIZE_PER_GPU
    * strategy.num_replicas_in_sync
)

CONFIG["batch_size_per_gpu"] = (
    VALIDATED_BATCH_SIZE_PER_GPU
)

CONFIG["global_batch_size"] = (
    VALIDATED_GLOBAL_BATCH_SIZE
)

CONFIG["benchmark_training_images_per_second"] = 630.06

print("=" * 60)
print("VALIDATED TRAINING CONFIGURATION")
print("=" * 60)

print(
    "GPUs                 :",
    strategy.num_replicas_in_sync
)

print(
    "Batch per GPU        :",
    CONFIG["batch_size_per_gpu"]
)

print(
    "Global batch         :",
    CONFIG["global_batch_size"]
)

print(
    "Learning rate        :",
    CONFIG["learning_rate"]
)

print(
    "Optimizer            :",
    CONFIG["optimizer"]
)

print(
    "Loss                 :",
    CONFIG["loss"]
)

print(
    "Epoch limit          :",
    CONFIG["epochs"]
)

print(
    "Benchmark throughput :",
    f"{CONFIG['benchmark_training_images_per_second']:.2f}",
    "images/sec"
)

print("=" * 60)

print("\n✓ Batch size validated on actual 2-GPU training")
print("✓ Configuration locked for the first 280K experiment")

VALIDATED TRAINING CONFIGURATION
GPUs                 : 2
Batch per GPU        : 32
Global batch         : 64
Learning rate        : 0.001
Optimizer            : Adam
Loss                 : mse
Epoch limit          : 50
Benchmark throughput : 630.06 images/sec

✓ Batch size validated on actual 2-GPU training
✓ Configuration locked for the first 280K experiment


In [18]:
# ============================================================
# CELL 18 — Checkpoint & Resume Configuration
# ============================================================

print("=" * 60)
print("CHECKPOINT & RESUME CONFIGURATION")
print("=" * 60)

# ------------------------------------------------------------
# Checkpoint directories
# ------------------------------------------------------------

LATEST_CHECKPOINT_DIR = os.path.join(
    DIRS["checkpoints"],
    "latest"
)

BEST_CHECKPOINT_DIR = os.path.join(
    DIRS["checkpoints"],
    "best"
)

os.makedirs(LATEST_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BEST_CHECKPOINT_DIR, exist_ok=True)


# ------------------------------------------------------------
# Persistent training-state files
# ------------------------------------------------------------

TRAINING_STATE_PATH = os.path.join(
    DIRS["logs"],
    "training_state.json"
)

HISTORY_PATH = os.path.join(
    DIRS["logs"],
    "history.json"
)

TRAINING_LOG_PATH = os.path.join(
    DIRS["logs"],
    "training.log"
)

CSV_LOG_PATH = os.path.join(
    DIRS["logs"],
    "training_log.csv"
)


# ------------------------------------------------------------
# Final model paths
# ------------------------------------------------------------

FINAL_AUTOENCODER_PATH = os.path.join(
    DIRS["models"],
    "AE_280K_final.keras"
)

FINAL_ENCODER_PATH = os.path.join(
    DIRS["models"],
    "AE_280K_encoder.keras"
)


# ------------------------------------------------------------
# Display paths
# ------------------------------------------------------------

print("Latest checkpoint directory:")
print(" ", LATEST_CHECKPOINT_DIR)

print("\nBest checkpoint directory:")
print(" ", BEST_CHECKPOINT_DIR)

print("\nTraining state:")
print(" ", TRAINING_STATE_PATH)

print("\nHistory:")
print(" ", HISTORY_PATH)

print("\nText log:")
print(" ", TRAINING_LOG_PATH)

print("\nCSV log:")
print(" ", CSV_LOG_PATH)

print("\nFinal autoencoder:")
print(" ", FINAL_AUTOENCODER_PATH)

print("\nFinal encoder:")
print(" ", FINAL_ENCODER_PATH)


# ------------------------------------------------------------
# Store paths in CONFIG
# ------------------------------------------------------------

CONFIG["latest_checkpoint_directory"] = (
    LATEST_CHECKPOINT_DIR
)

CONFIG["best_checkpoint_directory"] = (
    BEST_CHECKPOINT_DIR
)

CONFIG["training_state_path"] = (
    TRAINING_STATE_PATH
)

CONFIG["history_path"] = HISTORY_PATH

CONFIG["training_log_path"] = (
    TRAINING_LOG_PATH
)

CONFIG["csv_log_path"] = CSV_LOG_PATH

CONFIG["final_autoencoder_path"] = (
    FINAL_AUTOENCODER_PATH
)

CONFIG["final_encoder_path"] = (
    FINAL_ENCODER_PATH
)


print("\n" + "=" * 60)
print("✓ Checkpoint directories ready")
print("✓ Training-state paths ready")
print("✓ Final model paths ready")
print("=" * 60)

CHECKPOINT & RESUME CONFIGURATION
Latest checkpoint directory:
  /kaggle/working/AE_280K/checkpoints/latest

Best checkpoint directory:
  /kaggle/working/AE_280K/checkpoints/best

Training state:
  /kaggle/working/AE_280K/logs/training_state.json

History:
  /kaggle/working/AE_280K/logs/history.json

Text log:
  /kaggle/working/AE_280K/logs/training.log

CSV log:
  /kaggle/working/AE_280K/logs/training_log.csv

Final autoencoder:
  /kaggle/working/AE_280K/models/AE_280K_final.keras

Final encoder:
  /kaggle/working/AE_280K/models/AE_280K_encoder.keras

✓ Checkpoint directories ready
✓ Training-state paths ready
✓ Final model paths ready


In [19]:
# ============================================================
# CELL 19 — DURABLE CHECKPOINT CONFIGURATION
# ============================================================

import os
import json
from datetime import datetime

print("=" * 60)
print("DURABLE CHECKPOINT CONFIGURATION")
print("=" * 60)

# ------------------------------------------------------------
# IMPORTANT:
# /kaggle/working is temporary.
#
# We keep the working checkpoint AND create a dedicated
# export directory that we will explicitly package/download
# at the end of training.
# ------------------------------------------------------------

LATEST_CHECKPOINT_DIR = os.path.join(
    DIRS["checkpoints"], "latest"
)

BEST_CHECKPOINT_DIR = os.path.join(
    DIRS["checkpoints"], "best"
)

os.makedirs(LATEST_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(BEST_CHECKPOINT_DIR, exist_ok=True)

LATEST_MODEL_PATH = os.path.join(
    LATEST_CHECKPOINT_DIR,
    "latest.keras"
)

BEST_MODEL_PATH = os.path.join(
    BEST_CHECKPOINT_DIR,
    "best.keras"
)

TRAINING_STATE_PATH = os.path.join(
    DIRS["logs"],
    "training_state.json"
)

HISTORY_PATH = os.path.join(
    DIRS["logs"],
    "history.json"
)

CONFIG_PATH = os.path.join(
    DIRS["config"],
    "experiment_config.json"
)

# ------------------------------------------------------------
# Create a NEW state for this fresh training run.
# ------------------------------------------------------------

training_state = {
    "experiment_name": CONFIG["experiment_name"],
    "status": "not_started",
    "last_completed_epoch": 0,
    "best_val_loss": None,
    "best_epoch": None,
    "global_batch_size": CONFIG["global_batch_size"],
    "batch_size_per_gpu": CONFIG["batch_size_per_gpu"],
    "learning_rate": CONFIG["learning_rate"],
    "optimizer": CONFIG["optimizer"],
    "loss": CONFIG["loss"],
    "created_at": datetime.now().isoformat(),
    "updated_at": datetime.now().isoformat()
}

def save_training_state(state):

    state["updated_at"] = datetime.now().isoformat()

    temp_path = TRAINING_STATE_PATH + ".tmp"

    with open(temp_path, "w") as f:
        json.dump(state, f, indent=4)

    os.replace(
        temp_path,
        TRAINING_STATE_PATH
    )


def load_training_state():

    if not os.path.exists(TRAINING_STATE_PATH):
        return None

    with open(TRAINING_STATE_PATH, "r") as f:
        return json.load(f)


# ------------------------------------------------------------
# Save initial state
# ------------------------------------------------------------

save_training_state(
    training_state
)

# ------------------------------------------------------------
# Store paths in CONFIG
# ------------------------------------------------------------

CONFIG["latest_checkpoint_path"] = (
    LATEST_MODEL_PATH
)

CONFIG["best_checkpoint_path"] = (
    BEST_MODEL_PATH
)

CONFIG["training_state_path"] = (
    TRAINING_STATE_PATH
)

CONFIG["history_path"] = (
    HISTORY_PATH
)

# Save configuration
with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=4)

print("Latest checkpoint:")
print(" ", LATEST_MODEL_PATH)

print("\nBest checkpoint:")
print(" ", BEST_MODEL_PATH)

print("\nTraining state:")
print(" ", TRAINING_STATE_PATH)

print("\n✓ Checkpoint directories created")
print("✓ Training state initialized")
print("✓ Configuration saved")

print("=" * 60)

DURABLE CHECKPOINT CONFIGURATION
Latest checkpoint:
  /kaggle/working/AE_280K/checkpoints/latest/latest.keras

Best checkpoint:
  /kaggle/working/AE_280K/checkpoints/best/best.keras

Training state:
  /kaggle/working/AE_280K/logs/training_state.json

✓ Checkpoint directories created
✓ Training state initialized
✓ Configuration saved


In [20]:
# ============================================================
# CELL 20 — Rebuild Clean Model for Real Training
# ============================================================

print("=" * 60)
print("RESETTING MODEL FOR REAL 280K TRAINING")
print("=" * 60)

# ------------------------------------------------------------
# Rebuild the exact 10K architecture from scratch
# inside the same 2-GPU strategy.
# ------------------------------------------------------------

with strategy.scope():

    autoencoder, encoder, decoder = build_autoencoder()


# ------------------------------------------------------------
# Verify optimizer state is fresh
# ------------------------------------------------------------

optimizer_iterations = int(
    autoencoder.optimizer.iterations.numpy()
)

print("\nModel rebuilt successfully.")

print(
    "Optimizer iterations:",
    optimizer_iterations
)

print(
    "Total parameters    :",
    f"{autoencoder.count_params():,}"
)

# ------------------------------------------------------------
# Critical sanity checks
# ------------------------------------------------------------

assert optimizer_iterations == 0

assert autoencoder.count_params() == 1_079_108

# Verify output shape
test_shape = autoencoder.compute_output_shape(
    (None, 128, 128, 4)
)

print(
    "Model output shape   :",
    test_shape
)

assert tuple(test_shape) == (
    None,
    128,
    128,
    4
)

print("\n✓ Fresh model created")
print("✓ Optimizer state reset to iteration 0")
print("✓ Exact 1,079,108-parameter architecture confirmed")
print("✓ Input/output shape confirmed")
print("✓ Benchmark updates are NOT part of real training")


# ------------------------------------------------------------
# Save experiment configuration
# ------------------------------------------------------------

CONFIG_PATH = os.path.join(
    DIRS["config"],
    "experiment_config.json"
)

with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=4)

print(
    "\nExperiment configuration saved:"
)

print(
    CONFIG_PATH
)

print("=" * 60)

RESETTING MODEL FOR REAL 280K TRAINING

Model rebuilt successfully.
Optimizer iterations: 0
Total parameters    : 1,079,108
Model output shape   : (None, 128, 128, 4)

✓ Fresh model created
✓ Optimizer state reset to iteration 0
✓ Exact 1,079,108-parameter architecture confirmed
✓ Input/output shape confirmed
✓ Benchmark updates are NOT part of real training

Experiment configuration saved:
/kaggle/working/AE_280K/config/experiment_config.json


In [21]:
# ============================================================
# CELL 21 — DURABLE MODEL CHECKPOINT CALLBACKS
# ============================================================

import shutil
import numpy as np
from tensorflow import keras

print("=" * 60)
print("CREATING DURABLE CHECKPOINT CALLBACKS")
print("=" * 60)


class LatestCheckpointCallback(keras.callbacks.Callback):

    def on_epoch_end(self, epoch, logs=None):

        logs = logs or {}

        completed_epoch = epoch + 1

        # ----------------------------------------------------
        # Save complete model + optimizer state
        # ----------------------------------------------------

        self.model.save(
            LATEST_MODEL_PATH
        )

        # ----------------------------------------------------
        # ALSO create an easy-to-find copy
        # ----------------------------------------------------

        latest_export = os.path.join(
            DIRS["base"],
            "AE_280K_latest.keras"
        )

        shutil.copy2(
            LATEST_MODEL_PATH,
            latest_export
        )

        # ----------------------------------------------------
        # Update persistent state
        # ----------------------------------------------------

        training_state[
            "last_completed_epoch"
        ] = completed_epoch

        training_state["status"] = "training"

        save_training_state(
            training_state
        )

        print(
            f"\n[CHECKPOINT] Latest saved "
            f"after epoch {completed_epoch}"
        )

        print(
            "[CHECKPOINT] File size: "
            f"{os.path.getsize(LATEST_MODEL_PATH) / (1024**2):.2f} MB"
        )


class BestCheckpointCallback(keras.callbacks.Callback):

    def __init__(self):
        super().__init__()

        self.best = (
            np.inf
            if training_state["best_val_loss"] is None
            else float(
                training_state["best_val_loss"]
            )
        )

    def on_epoch_end(self, epoch, logs=None):

        logs = logs or {}

        current_val_loss = logs.get(
            "val_loss"
        )

        if current_val_loss is None:
            return

        current_val_loss = float(
            current_val_loss
        )

        completed_epoch = epoch + 1

        if current_val_loss < self.best:

            self.best = current_val_loss

            # ------------------------------------------------
            # Save complete best model
            # ------------------------------------------------

            self.model.save(
                BEST_MODEL_PATH
            )

            # ------------------------------------------------
            # Easy-to-find copy
            # ------------------------------------------------

            best_export = os.path.join(
                DIRS["base"],
                "AE_280K_best.keras"
            )

            shutil.copy2(
                BEST_MODEL_PATH,
                best_export
            )

            # ------------------------------------------------
            # Update state
            # ------------------------------------------------

            training_state[
                "best_val_loss"
            ] = current_val_loss

            training_state[
                "best_epoch"
            ] = completed_epoch

            save_training_state(
                training_state
            )

            print(
                f"\n[BEST] Epoch {completed_epoch}: "
                f"val_loss = "
                f"{current_val_loss:.10f}"
            )

            print(
                "[BEST] File size: "
                f"{os.path.getsize(BEST_MODEL_PATH) / (1024**2):.2f} MB"
            )


class TrainingStateCallback(
    keras.callbacks.Callback
):

    def on_train_begin(self, logs=None):

        training_state["status"] = "training"

        save_training_state(
            training_state
        )

    def on_train_end(self, logs=None):

        if (
            training_state[
                "last_completed_epoch"
            ]
            >= CONFIG["epochs"]
        ):

            training_state[
                "status"
            ] = "completed"

        else:

            training_state[
                "status"
            ] = "stopped"

        save_training_state(
            training_state
        )


# ------------------------------------------------------------
# Create callbacks
# ------------------------------------------------------------

latest_checkpoint_callback = (
    LatestCheckpointCallback()
)

best_checkpoint_callback = (
    BestCheckpointCallback()
)

training_state_callback = (
    TrainingStateCallback()
)

CALLBACKS = [
    training_state_callback,
    latest_checkpoint_callback,
    best_checkpoint_callback,
]


print("Callbacks created:")
print("  ✓ Training state")
print("  ✓ Latest checkpoint")
print("  ✓ Best checkpoint")

print("\nEvery epoch:")
print("  ✓ latest.keras saved")
print("  ✓ latest export copy created")
print("  ✓ training_state.json updated")

print("\nEvery validation improvement:")
print("  ✓ best.keras saved")
print("  ✓ best export copy created")
print("  ✓ best validation metadata updated")

print("=" * 60)

CREATING DURABLE CHECKPOINT CALLBACKS
Callbacks created:
  ✓ Training state
  ✓ Latest checkpoint
  ✓ Best checkpoint

Every epoch:
  ✓ latest.keras saved
  ✓ latest export copy created
  ✓ training_state.json updated

Every validation improvement:
  ✓ best.keras saved
  ✓ best export copy created
  ✓ best validation metadata updated


Training

In [22]:
# ============================================================
# CELL 22 — FINAL 280K TRAINING
# ============================================================

print("=" * 60)
print("STARTING FINAL 280K TRAINING")
print("=" * 60)

# ------------------------------------------------------------
# Build datasets
# ------------------------------------------------------------

train_ds = prepare_dataset(
    train_raw,
    batch_size=CONFIG["global_batch_size"],
    training=True
)

val_ds = prepare_dataset(
    val_raw,
    batch_size=CONFIG["global_batch_size"],
    training=False
)

print("Train dataset : ready")
print("Val dataset   : ready")
print("Global batch  :", CONFIG["global_batch_size"])


# ------------------------------------------------------------
# IMPORTANT:
# This is a NEW run.
# Cell 20 already created the clean model.
# ------------------------------------------------------------

print("\nStarting from clean model")

print(
    "Optimizer iterations:",
    int(autoencoder.optimizer.iterations.numpy())
)

assert int(
    autoencoder.optimizer.iterations.numpy()
) == 0


# ------------------------------------------------------------
# Fresh callbacks
# ------------------------------------------------------------

latest_checkpoint_callback = (
    LatestCheckpointCallback()
)

best_checkpoint_callback = (
    BestCheckpointCallback()
)

training_state_callback = (
    TrainingStateCallback()
)

CALLBACKS = [
    training_state_callback,
    latest_checkpoint_callback,
    best_checkpoint_callback,
]


# ------------------------------------------------------------
# TRAIN
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("TRAINING")
print("=" * 60)

history = autoencoder.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG["epochs"],
    callbacks=CALLBACKS,
    verbose=1
)


# ------------------------------------------------------------
# FINAL SAVE
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL MODEL SAVE")
print("=" * 60)

FINAL_AUTOENCODER_PATH = os.path.join(
    DIRS["models"],
    "AE_280K_final.keras"
)

FINAL_BEST_PATH = os.path.join(
    DIRS["models"],
    "AE_280K_best.keras"
)

FINAL_ENCODER_PATH = os.path.join(
    DIRS["models"],
    "AE_280K_encoder.keras"
)


# Save final epoch-50 model
autoencoder.save(
    FINAL_AUTOENCODER_PATH
)

# Save encoder
encoder = autoencoder.get_layer("encoder")

encoder.save(
    FINAL_ENCODER_PATH
)


# Copy the best checkpoint into models/
if os.path.exists(BEST_MODEL_PATH):

    shutil.copy2(
        BEST_MODEL_PATH,
        FINAL_BEST_PATH
    )


# ------------------------------------------------------------
# Save history
# ------------------------------------------------------------

history_data = {
    key: [float(v) for v in values]
    for key, values in history.history.items()
}

with open(HISTORY_PATH, "w") as f:
    json.dump(
        history_data,
        f,
        indent=4
    )


# ------------------------------------------------------------
# Final state
# ------------------------------------------------------------

training_state["status"] = "completed"

training_state[
    "last_completed_epoch"
] = CONFIG["epochs"]

save_training_state(
    training_state
)


# ------------------------------------------------------------
# VERIFY FILES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("VERIFYING FINAL ARTIFACTS")
print("=" * 60)

required_files = [
    FINAL_AUTOENCODER_PATH,
    FINAL_ENCODER_PATH,
    HISTORY_PATH,
    TRAINING_STATE_PATH,
]

if os.path.exists(FINAL_BEST_PATH):
    required_files.append(
        FINAL_BEST_PATH
    )

for path in required_files:

    exists = os.path.exists(path)

    print(
        f"{'✓' if exists else '✗'} "
        f"{path}"
    )

    if exists:

        size_mb = (
            os.path.getsize(path)
            / (1024 * 1024)
        )

        print(
            f"    Size: {size_mb:.2f} MB"
        )

        assert size_mb > 0


print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)

print(
    "Completed epochs:",
    training_state["last_completed_epoch"]
)

print(
    "Best epoch:",
    training_state["best_epoch"]
)

print(
    "Best val_loss:",
    training_state["best_val_loss"]
)

print(
    "Final optimizer iterations:",
    int(autoencoder.optimizer.iterations.numpy())
)

print("=" * 60)

STARTING FINAL 280K TRAINING
Train dataset : ready
Val dataset   : ready
Global batch  : 64

Starting from clean model
Optimizer iterations: 0

TRAINING
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
Epoch 1/50
INFO:tensorflow:Collective all_reduce tensors: 16 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
   3970/Unknown 316s 78ms/step - loss: 0.0192INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

[CHECKPOINT] Latest saved after epoch 1
[CHECKPOINT] File size: 12.41 MB

[BEST] Epoch 1: val_loss = 0.0037864617
[BEST] File size: 12.41 MB
3972/3972 ━━━━━━━━━━━━━━━━━━━━ 335s 83ms/step - loss: 0.0086 - val_loss: 0.0038
Epoch 2/50
3972/3972 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - loss: 0.0039
[CHECKPOINT] Latest saved after epoch 2
[CHECKPOINT] File size: 12.41 MB

[BEST] Epoch 2: val_loss = 0.0028494573
[BEST] File size: 12.41 MB
3972/39

In [23]:
import os

print("Current /kaggle/working contents:")
print(os.listdir("/kaggle/working"))

print("\nAE_280K exists:")
print(os.path.exists("/kaggle/working/AE_280K"))

Current /kaggle/working contents:
['.virtual_documents', 'AE_280K']

AE_280K exists:
True


In [24]:
import os

BASE_DIR = "/kaggle/working/AE_280K"

for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = "  " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        path = os.path.join(root, file)
        size_mb = os.path.getsize(path) / (1024 * 1024)

        print(
            f"{indent}  {file} "
            f"({size_mb:.2f} MB)"
        )

AE_280K/
  AE_280K_best.keras (12.41 MB)
  AE_280K_latest.keras (12.41 MB)
  checkpoints/
    latest/
      latest.keras (12.41 MB)
    best/
      best.keras (12.41 MB)
  samples/
  logs/
    training_state.json (0.00 MB)
    history.json (0.00 MB)
  evaluation/
  models/
    AE_280K_encoder.keras (1.50 MB)
    AE_280K_best.keras (12.41 MB)
    AE_280K_final.keras (12.41 MB)
  config/
    val_indices.json (0.10 MB)
    dataset_manifest.json (0.00 MB)
    train_indices.json (1.85 MB)
    unique_indices.json (4.21 MB)
    experiment_config.json (0.00 MB)
    shuffled_unique_indices.json (4.21 MB)
    test_indices.json (0.10 MB)


In [25]:
# ============================================================
# RECOVER ORIGINAL TRAINING STATE
# ============================================================

STATE_PATH = "/kaggle/working/AE_280K/logs/training_state.json"

with open(STATE_PATH, "r") as f:
    state = json.load(f)

print("=" * 60)
print("SAVED TRAINING STATE")
print("=" * 60)

for key, value in state.items():
    print(f"{key:30s}: {value}")

print("=" * 60)

SAVED TRAINING STATE
experiment_name               : AE_280K_full
status                        : completed
last_completed_epoch          : 50
best_val_loss                 : 0.00045805147965438664
best_epoch                    : 49
global_batch_size             : 64
batch_size_per_gpu            : 32
learning_rate                 : 0.001
optimizer                     : Adam
loss                          : mse
created_at                    : 2026-08-19T04:53:41.612180
updated_at                    : 2026-08-19T09:51:26.413034


In [26]:
# ============================================================
# RECOVERY — SEARCH FOR ALL KERAS MODEL FILES
# ============================================================

import os

print("=" * 60)
print("SEARCHING FOR SAVED MODEL CHECKPOINTS")
print("=" * 60)

search_roots = [
    "/kaggle/working",
    "/kaggle/input",
    "/tmp",
]

found = []

for search_root in search_roots:

    if not os.path.exists(search_root):
        continue

    for root, dirs, files in os.walk(search_root):

        for filename in files:

            if filename.endswith(
                (".keras", ".h5", ".hdf5")
            ):

                path = os.path.join(
                    root,
                    filename
                )

                try:
                    size_mb = (
                        os.path.getsize(path)
                        / (1024 * 1024)
                    )

                    found.append(
                        (path, size_mb)
                    )

                except OSError:
                    pass


if found:

    print(
        f"\nFOUND {len(found)} MODEL FILE(S):\n"
    )

    for path, size_mb in found:

        print(
            f"{size_mb:10.2f} MB  {path}"
        )

else:

    print(
        "\nNO .keras / .h5 / .hdf5 MODEL FILES FOUND."
    )

print("=" * 60)

SEARCHING FOR SAVED MODEL CHECKPOINTS

FOUND 7 MODEL FILE(S):

     12.41 MB  /kaggle/working/AE_280K/AE_280K_best.keras
     12.41 MB  /kaggle/working/AE_280K/AE_280K_latest.keras
     12.41 MB  /kaggle/working/AE_280K/checkpoints/latest/latest.keras
     12.41 MB  /kaggle/working/AE_280K/checkpoints/best/best.keras
      1.50 MB  /kaggle/working/AE_280K/models/AE_280K_encoder.keras
     12.41 MB  /kaggle/working/AE_280K/models/AE_280K_best.keras
     12.41 MB  /kaggle/working/AE_280K/models/AE_280K_final.keras


In [27]:
# ============================================================
# CELL 23 — Save Training History
# ============================================================

print("=" * 60)
print("SAVING TRAINING HISTORY")
print("=" * 60)

# ------------------------------------------------------------
# Convert Keras History object to plain Python dictionary
# ------------------------------------------------------------

history_data = {}

if hasattr(history, "history"):
    history_data = {
        key: [float(value) for value in values]
        for key, values in history.history.items()
    }

# ------------------------------------------------------------
# Save history as JSON
# ------------------------------------------------------------

with open(HISTORY_PATH, "w") as f:
    json.dump(history_data, f, indent=4)

# ------------------------------------------------------------
# Update CONFIG with final training information
# ------------------------------------------------------------

CONFIG["completed_epochs"] = (
    training_state["last_completed_epoch"]
)

CONFIG["best_epoch"] = (
    training_state["best_epoch"]
)

CONFIG["best_val_loss"] = (
    training_state["best_val_loss"]
)

CONFIG["final_optimizer_iterations"] = int(
    autoencoder.optimizer.iterations.numpy()
)

# Save updated configuration
with open(CONFIG_PATH, "w") as f:
    json.dump(CONFIG, f, indent=4)

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print(
    f"Epochs recorded : "
    f"{len(history_data.get('loss', []))}"
)

print(
    f"Best epoch      : "
    f"{CONFIG['best_epoch']}"
)

print(
    f"Best val_loss   : "
    f"{CONFIG['best_val_loss']:.10f}"
)

print(
    f"Final epoch     : "
    f"{CONFIG['completed_epochs']}"
)

print(
    f"History saved   : "
    f"{HISTORY_PATH}"
)

print(
    f"Config updated  : "
    f"{CONFIG_PATH}"
)

print("\n✓ Training history saved")
print("✓ Experiment configuration updated")

print("=" * 60)

SAVING TRAINING HISTORY
Epochs recorded : 50
Best epoch      : 49
Best val_loss   : 0.0004580515
Final epoch     : 50
History saved   : /kaggle/working/AE_280K/logs/history.json
Config updated  : /kaggle/working/AE_280K/config/experiment_config.json

✓ Training history saved
✓ Experiment configuration updated


In [28]:
# ============================================================
# CELL 24 — Restore Completed 280K Experiment
# ============================================================

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras

print("=" * 60)
print("RESTORING COMPLETED 280K EXPERIMENT")
print("=" * 60)

# ------------------------------------------------------------
# Base paths
# ------------------------------------------------------------

BASE_DIR = "/kaggle/working/AE_280K"

DIRS = {
    "base": BASE_DIR,
    "checkpoints": os.path.join(BASE_DIR, "checkpoints"),
    "models": os.path.join(BASE_DIR, "models"),
    "logs": os.path.join(BASE_DIR, "logs"),
    "evaluation": os.path.join(BASE_DIR, "evaluation"),
    "config": os.path.join(BASE_DIR, "config"),
    "samples": os.path.join(BASE_DIR, "samples"),
}

TRAINING_STATE_PATH = os.path.join(
    DIRS["logs"], "training_state.json"
)

CONFIG_PATH = os.path.join(
    DIRS["config"], "experiment_config.json"
)

DATASET_MANIFEST_PATH = os.path.join(
    DIRS["config"], "dataset_manifest.json"
)

TRAIN_MANIFEST_PATH = os.path.join(
    DIRS["config"], "train_indices.json"
)

VAL_MANIFEST_PATH = os.path.join(
    DIRS["config"], "val_indices.json"
)

TEST_MANIFEST_PATH = os.path.join(
    DIRS["config"], "test_indices.json"
)

BEST_MODEL_PATH = os.path.join(
    DIRS["checkpoints"],
    "best",
    "best.keras"
)

LATEST_MODEL_PATH = os.path.join(
    DIRS["checkpoints"],
    "latest",
    "latest.keras"
)

# ------------------------------------------------------------
# Load saved metadata
# ------------------------------------------------------------

with open(CONFIG_PATH, "r") as f:
    CONFIG = json.load(f)

with open(TRAINING_STATE_PATH, "r") as f:
    training_state = json.load(f)

with open(DATASET_MANIFEST_PATH, "r") as f:
    dataset_manifest = json.load(f)

# ------------------------------------------------------------
# Verify completed training
# ------------------------------------------------------------

print("Training status       :", training_state["status"])
print(
    "Last completed epoch :",
    training_state["last_completed_epoch"]
)
print(
    "Best epoch            :",
    training_state["best_epoch"]
)
print(
    "Best validation loss  :",
    training_state["best_val_loss"]
)

assert training_state["last_completed_epoch"] == 50
assert training_state["best_epoch"] == 32
assert os.path.exists(BEST_MODEL_PATH)
assert os.path.exists(LATEST_MODEL_PATH)

# ------------------------------------------------------------
# Load BEST model
# ------------------------------------------------------------

print("\nLoading BEST model:")
print(BEST_MODEL_PATH)

autoencoder = keras.models.load_model(
    BEST_MODEL_PATH
)

# ------------------------------------------------------------
# Recover encoder
# ------------------------------------------------------------

encoder = autoencoder.get_layer("encoder")

print("\n" + "=" * 60)
print("RESTORE COMPLETE")
print("=" * 60)

print(
    "Model parameters :",
    f"{autoencoder.count_params():,}"
)

print(
    "Best epoch       :",
    training_state["best_epoch"]
)

print(
    "Best val_loss    :",
    f"{training_state['best_val_loss']:.10f}"
)

print(
    "Model loaded from:",
    "BEST checkpoint"
)

print("\n✓ Experiment metadata restored")
print("✓ Dataset manifests located")
print("✓ BEST model restored")
print("✓ Encoder restored")
print("✓ No training performed")

print("=" * 60)

RESTORING COMPLETED 280K EXPERIMENT
Training status       : completed
Last completed epoch : 50
Best epoch            : 49
Best validation loss  : 0.00045805147965438664


AssertionError: 

In [2]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

project_dir = Path("/kaggle/working/AE_280K")
zip_base = Path("/kaggle/working/AE_280K_COMPLETE_BACKUP")

# Create one ZIP containing the entire project
shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=project_dir.parent,
    base_dir=project_dir.name
)

zip_file = Path(str(zip_base) + ".zip")

print("=" * 60)
print("COMPLETE PROJECT BACKUP CREATED")
print("=" * 60)
print(f"File: {zip_file}")
print(f"Size: {zip_file.stat().st_size / (1024**3):.2f} GB")
print("=" * 60)

display(FileLink(str(zip_file)))

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/AE_280K'

In [4]:
import os

print("Searching /kaggle/working ...")

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith((".keras", ".json", ".csv", ".log")):
            path = os.path.join(root, file)
            size = os.path.getsize(path) / (1024**2)
            print(f"{size:8.2f} MB  {path}")

Searching /kaggle/working ...
